# Parse predicted labels

In [1]:
import numpy as np
import pandas as pd

### Load the questions and gold labels

In [2]:
df = pd.read_csv("../Datasets/MedChangeQA.csv")

questions = df["Question"].astype(str).str.strip().tolist()
labels_n = df["Newest Label"].astype(str).str.strip().tolist()
labels_o = df["Outdated Label"].astype(str).str.strip().tolist()

len(questions)

512

In [3]:
def parse_labels(lines):
    # copied unchanged from get_predictions() in the authors' experiments_results.ipynb
    llm_predicted_labels = list()
    for line in lines:
        if "SUPPORTED" in line:
            llm_predicted_labels.append("SUPPORTED")
        elif "REFUTED" in line:
            llm_predicted_labels.append("REFUTED")
        elif "NOT ENOUGH INFORMATION" in line:
            llm_predicted_labels.append("NOT ENOUGH INFORMATION")
    return llm_predicted_labels

### Parse each model's answers

In [4]:
# generated answers of our models (one line per MedChangeQA question, in CSV order).

filenames = {
    "qwen25-7b": "output/qwen25-7b_answers.txt",
    "biomistral-7b": "output/biomistral-7b_answers.txt",
    "biomistral-7b (new)": "output/biomistral-7b_answers_(new)",
}

LABELS = ["SUPPORTED", "REFUTED", "NOT ENOUGH INFORMATION"]

parsed = {}
for model, filename in filenames.items():
    with open(filename, "r", encoding="utf-8") as f:
        lines = f.readlines()

    predictions = parse_labels(lines)

    unparsed = [i for i, line in enumerate(lines) if not any(l in line for l in LABELS)]
    multi = [i for i, line in enumerate(lines) if sum(l in line for l in LABELS) > 1]

    print(f"{model}: {len(lines)} answers -> {len(predictions)} labels")
    print(f"  answers with no label (skipped by the rule): {unparsed}")
    print(f"  answers mentioning more than one label (first match in rule order wins): {multi}")
    print(f"  label counts: {pd.Series(predictions).value_counts().to_dict()}")

    if len(predictions) == len(questions):
        parsed[model] = predictions
    else:
        print("  !! number of labels does not match the 512 questions -> not aligned, left out")
    print()

qwen25-7b: 512 answers -> 512 labels
  answers with no label (skipped by the rule): []
  answers mentioning more than one label (first match in rule order wins): [344]
  label counts: {'SUPPORTED': 379, 'NOT ENOUGH INFORMATION': 114, 'REFUTED': 19}

biomistral-7b: 512 answers -> 503 labels
  answers with no label (skipped by the rule): [61, 103, 118, 328, 371, 387, 440, 494, 509]
  answers mentioning more than one label (first match in rule order wins): []
  label counts: {'NOT ENOUGH INFORMATION': 464, 'SUPPORTED': 38, 'REFUTED': 1}
  !! number of labels does not match the 512 questions -> not aligned, left out

biomistral-7b (new): 512 answers -> 512 labels
  answers with no label (skipped by the rule): []
  answers mentioning more than one label (first match in rule order wins): []
  label counts: {'SUPPORTED': 261, 'NOT ENOUGH INFORMATION': 251}



### Save the parsed labels

In [5]:
out = df[["Question", "Newest Label", "Outdated Label"]].copy()
for model, predictions in parsed.items():
    out[model] = predictions

out.to_csv("output/parsed_labels.csv", index=False)
out.head()

,Question,Newest Label,Outdated Label,qwen25-7b,biomistral-7b (new)
0,Is cranberry juice effective for treating urin...,SUPPORTED,REFUTED,SUPPORTED,SUPPORTED
1,Is add-on lamotrigine effective in reducing se...,SUPPORTED,NOT ENOUGH INFORMATION,SUPPORTED,SUPPORTED
2,Do statins help reduce cardiovascular events i...,SUPPORTED,NOT ENOUGH INFORMATION,SUPPORTED,SUPPORTED
3,Do antibiotics effectively reduce pain in chil...,SUPPORTED,NOT ENOUGH INFORMATION,SUPPORTED,NOT ENOUGH INFORMATION
4,Does cell salvage reduce the need for blood tr...,REFUTED,NOT ENOUGH INFORMATION,SUPPORTED,SUPPORTED
